# AIC SkillSelect ETL Notebook
**Australian Centre of English — Market Intelligence Project**

This notebook is used as a control panel for running and checking the SkillSelect ETL workflow.

The main automation logic is stored in:

`ETL/skillselect_csv_exporter.py`

The Jupyter notebook is mainly used to run the script, preview exported CSV files, and later check the database output.

---

## Architecture

The Python script works like a human user using the SkillSelect dashboard:

1. Open the SkillSelect public dashboard  
2. Navigate to the EOI Parameters page  
3. Select two additional columns using the Yes/No tiles  
4. Click NEXT to open the Results Table  
5. Right-click the table and export the data as CSV  
6. Save the downloaded CSV file  
7. Repeat the same process for each selected column pair  
8. Later, transform the CSV files and load them into SQLite for analysis

In simple terms:

**Open website → select columns → go to results → right-click export → save CSV → repeat → prepare for database loading**

---

## Run cells in this order

### 1. Setup
Install and check the required Python packages.

Main packages used:
- `playwright`
- `pandas`
- `sqlite3` / SQLite
- `pathlib`

---

### 2. Export CSVs
Run the Playwright automation script.

This script opens the browser, clicks through the SkillSelect dashboard, selects the required column pairs, exports the results table as CSV, and saves the files into:

`raw_data/skillselect_exports`

The script uses text and role-based locators where possible instead of relying on Qlik CSS classes, because Qlik class names are dynamic and can change.

---

### 3. Preview
Check the exported CSV files.

This step confirms that:
- the files were downloaded successfully
- the columns look correct
- the data can be opened with pandas
- the exported files are ready for transformation

---

### 4. ETL
Transform the exported CSV files into a cleaner structure for database use.

The expected process is:

**CSV files → cleaned tables → long-format structure → load into SQLite**

This step prepares the data for analysis and dashboard development.

---

### 5. Verify
Check the final database output.

This step confirms that:
- data has been loaded correctly
- row counts are reasonable
- key fields are available
- the database is ready for reporting and analysis


## Cell 1 — Install dependencies

This cell installs the Python packages required for the SkillSelect ETL workflow.

It installs:
- `playwright` for browser automation
- `pandas` for reading and checking CSV files
- `sqlalchemy` for future database connection support
- `openpyxl` for Excel compatibility if needed later

This cell only needs to be run when setting up the environment for the first time, or when using a new Python/Anaconda environment.

In [1]:
import subprocess, sys

packages = ['playwright', 'pandas', 'sqlalchemy', 'openpyxl']

for pkg in packages:
    subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'], check=True)

subprocess.run([sys.executable, '-m', 'playwright', 'install', 'chromium'], check=True)

print('✅ All packages installed')

✅ All packages installed


## Cell 2 — Run SkillSelect CSV export

This cell runs the main SkillSelect exporter script:

`ETL/skillselect_csv_exporter.py`

The script uses Playwright to automate the public SkillSelect dashboard.

It works like a human user:
1. Opens the SkillSelect dashboard
2. Goes to the EOI Parameters page
3. Selects two additional column tiles
4. Clicks NEXT to open the Results Table
5. Right-clicks the results table
6. Selects the export/download option
7. Saves the CSV file
8. Repeats the same process for all selected column pairs

This cell runs all 7 selected column-pair exports.

In [2]:
%cd ~/Documents/Gov_ETL_data
!python ETL/skillselect_csv_exporter.py --pairs 1 2 3 4 5 6 7

🚀 Opening SkillSelect...
✅ Page loaded

[1/7] Occupations × Points
    [select] 'Visa Type' → '189'
  ❌ Occupations×Points: Locator.wait_for: Timeout 10000ms exceeded.
Call log:
  - waiting for get_by_text("Visa Type").first to be visible
    24 × locator resolved to hidden <a class="qcmd" data-qcmd="navClick" data-pagesource="https://api.dynamic.reports.employment.gov.au/anonap/single/?appid=aaac76b5-ad30-477e-9ca0-472f8ab57fc8&sheet=1fbfd90f-e36c-44b9-a078-a7c78a46792c&opt=ctxmenu">Dashboard Results Table: EOI by Visa Type and Sta…</a>


[2/7] Occupations × English Test Score
    [select] 'Visa Type' → '189'
  ❌ Occupations×English Test Score: Locator.wait_for: Timeout 10000ms exceeded.
Call log:
  - waiting for get_by_text("Visa Type").first to be visible
    24 × locator resolved to hidden <a class="qcmd" data-qcmd="navClick" data-pagesource="https://api.dynamic.reports.employment.gov.au/anonap/single/?appid=aaac76b5-ad30-477e-9ca0-472f8ab57fc8&sheet=1fbfd90f-e36c-44b9-a078-a7c78a4

## Cell 3 — Check exported files

This cell lists all CSV files saved in the SkillSelect export folder.

The purpose is to confirm that the export process created the expected files.

Expected folder:

`raw_data/skillselect_exports`

If the export worked correctly, this folder should contain CSV files for the selected SkillSelect column pairs.

In [ ]:
!find raw_data/skillselect_exports -type f | sort

## Cell 4 — Preview exported CSV files

This cell opens each exported CSV file using pandas.

It shows:
- the file name
- the number of rows and columns
- the first few rows of data

This helps confirm that the CSV files were downloaded correctly and can be used for the next ETL step.

In [ ]:
import pandas as pd
from pathlib import Path

files = sorted(Path("raw_data/skillselect_exports").glob("*.csv"))

for f in files:
    df = pd.read_csv(f)
    print("\n" + "="*80)
    print(f.name)
    print("Shape:", df.shape)
    display(df.head())

## Cell 5 — Basic data quality check

This cell performs a simple quality check on the exported CSV files.

It checks:
- column names
- number of rows
- missing values
- whether each file can be read successfully

This is not the full ETL process yet. It is a quick validation step before transforming and loading the data into SQLite.

In [ ]:
for f in files:
    df = pd.read_csv(f)

    print("\n" + "="*80)
    print(f.name)
    print("Rows:", len(df))
    print("Columns:", list(df.columns))
    print("\nMissing values:")
    print(df.isna().sum())

## Cell 6 — Monthly SkillSelect refresh

ใช้รันเมื่อต้องการ refresh ข้อมูล เช่น รายเดือน

Run this cell when the SkillSelect data needs to be refreshed.

This command runs all 7 selected SkillSelect column-pair exports and saves the CSV files into:

`raw_data/skillselect_exports`

This is currently a manual monthly refresh process. It can be converted into an automatic scheduled ETL job later after the MySQL loading script has been completed and tested.

In [ ]:
%cd ~/Documents/Gov_ETL_data
!python ETL/skillselect_csv_exporter.py --pairs 1 2 3 4 5 6 7

# Cell 6 — Future ETL to SQLite

This section will be used later to transform the exported SkillSelect CSV files and load them into a SQLite database.

Planned ETL process:

1. Read all exported CSV files
2. Clean column names and data types
3. Standardise values such as dates, visa types, EOI status, and count fields
4. Convert the files into a database-ready structure
5. Load the cleaned data into SQLite
6. Verify row counts and table structure

This cell is a placeholder for the next stage of the project.